# Fertilizer Predictor (K-Fold Ensemble)

This notebook defines and demonstrates a `FertilizerPredictorKFold` class. This class loads the k-fold trained base models (LightGBM, CatBoost, XGBoost), the meta-model (Logistic Regression), and all necessary preprocessing artifacts that were saved by the `Training.ipynb` notebook.

The predictor can:
1. Load all required models and artifacts from the `trained_models_kfold` directory.
2. Preprocess new input data (single instance or DataFrame) using the same transformations applied during training.
3. Generate predictions by:
    a. Getting averaged probability predictions from the k-fold models of each base classifier type.
    b. Using these averaged probabilities as input to the trained meta-model.
    c. Outputting the final fertilizer prediction(s).
4. Provide the top `k` fertilizer predictions along with their probabilities.

## 1. Setup and Imports

In [ ]:
import pandas as pd
import numpy as np
import joblib
import os
from sklearn.preprocessing import LabelEncoder, StandardScaler # For loading
import warnings

warnings.filterwarnings('ignore')
np.random.seed(42) # Consistent behavior if any random ops were part of it, though unlikely for predictor

## 2. Utility Functions (from Training Notebook)

We need the `DataFrameColumnEncoder` class definition and the `feature_engineering` function to be available for the predictor to correctly process new data. These should be identical to the ones used in `Training.ipynb`.

In [ ]:
class DataFrameColumnEncoder:
    """Custom encoder for categorical columns using LabelEncoder."""
    def __init__(self, columns_to_encode):
        self.columns_to_encode = columns_to_encode
        self.encoders_ = {col: LabelEncoder() for col in self.columns_to_encode}
        self.fitted_columns_ = []

    def fit(self, X, y=None):
        self.fitted_columns_ = [col for col in self.columns_to_encode if col in X.columns]
        for col in self.fitted_columns_:
            self.encoders_[col].fit(X[col].astype(str))
        return self

    def transform(self, X):
        X_transformed = X.copy()
        for col in self.fitted_columns_:
            # Attempt to transform, new values will raise error if encoder isn't robust to it
            # For prediction, it's crucial that new values are handled or models can accept them
            # The saved encoders from training will define known categories.
            # Models like CatBoost/LightGBM might handle new categories if configured.
            # Here, we rely on the encoder's behavior (error for unknown unless classes were augmented).
            try:
                X_transformed[col] = self.encoders_[col].transform(X_transformed[col].astype(str))
            except ValueError as e:
                # This is a critical point for prediction on unseen data.
                # If new categories appear, how should they be handled?
                # Option 1: Error out (current behavior of LabelEncoder by default)
                # Option 2: Assign a specific 'unknown' category index if the model was trained with it.
                # Option 3: If models are tree-based, they might have internal handling for unknowns.
                print(f"Warning: Encountered new value in column '{col}' during transform. This may lead to errors or unexpected behavior if models were not trained to handle unknowns. Error: {e}")
                # For now, let the error propagate or be handled by the model. A robust solution would define this.
                # A simple fallback for demonstration: assign -1 or a specific code, but this must align with training.
                # X_transformed[col] = -1 # Example: if -1 was an 'unknown' category index
                pass 
        return X_transformed
    
    # Required for scikit-learn compatibility if used in a Pipeline later (though not directly here)
    def get_params(self, deep=True):
        return {"columns_to_encode": self.columns_to_encode}

    def set_params(self, **params):
        for key, value in params.items():
            setattr(self, key, value)
        return self

def feature_engineering(df_input):
    """Apply feature engineering to the dataset. Must be identical to training."""
    df = df_input.copy()
    epsilon = 1e-6
    numerical_cols = ['Temparature', 'Humidity', 'Moisture', 'Nitrogen', 'Potassium', 'Phosphorous']
    
    if 'Nitrogen' in df.columns and 'Phosphorous' in df.columns:
        df['N_P_ratio'] = df['Nitrogen'] / (df['Phosphorous'] + epsilon)
    if 'Nitrogen' in df.columns and 'Potassium' in df.columns:
        df['N_K_ratio'] = df['Nitrogen'] / (df['Potassium'] + epsilon)
    if 'Phosphorous' in df.columns and 'Potassium' in df.columns:
        df['P_K_ratio'] = df['Phosphorous'] / (df['Potassium'] + epsilon)
    
    for col in numerical_cols:
        if col in df.columns:
            df[f'log_{col}'] = np.log1p(df[col])
            df[f'sq_{col}'] = df[col]**2
    
    if 'Temparature' in df.columns and 'Humidity' in df.columns:
        df['Temp_Hum_Interaction'] = df['Temparature'] * df['Humidity']
    if 'Moisture' in df.columns and 'Temparature' in df.columns:
        df['Moisture_Temp_Interaction'] = df['Moisture'] * df['Temparature']
        
    return df

## 3. FertilizerPredictorKFold Class

In [ ]:
class FertilizerPredictorKFold:
    def __init__(self, model_dir='trained_models_kfold'):
        self.model_dir = model_dir
        self.lgbm_kfold_models = []
        self.catboost_kfold_models = []
        self.xgb_kfold_models = []
        self.meta_model = None
        self.stacking_scaler = None
        self.column_encoder = None
        self.target_encoder = None
        self.feature_names = None
        self.is_loaded = False
        self._load_artifacts()

    def _load_artifacts(self):
        print(f"Loading artifacts from: {self.model_dir}")
        try:
            self.lgbm_kfold_models = joblib.load(os.path.join(self.model_dir, 'lgbm_kfold_models.joblib'))
            self.catboost_kfold_models = joblib.load(os.path.join(self.model_dir, 'catboost_kfold_models.joblib'))
            self.xgb_kfold_models = joblib.load(os.path.join(self.model_dir, 'xgboost_kfold_models.joblib'))
            self.meta_model = joblib.load(os.path.join(self.model_dir, 'meta_model_logistic_regression.joblib'))
            self.stacking_scaler = joblib.load(os.path.join(self.model_dir, 'stacking_scaler.joblib'))
            self.column_encoder = joblib.load(os.path.join(self.model_dir, 'dataframe_column_encoder.joblib'))
            self.target_encoder = joblib.load(os.path.join(self.model_dir, 'target_label_encoder.joblib'))
            self.feature_names = joblib.load(os.path.join(self.model_dir, 'feature_names.joblib'))
            self.is_loaded = True
            print("All models and artifacts loaded successfully.")
            print(f"  Loaded {len(self.lgbm_kfold_models)} LGBM models.")
            print(f"  Loaded {len(self.catboost_kfold_models)} CatBoost models.")
            print(f"  Loaded {len(self.xgb_kfold_models)} XGBoost models.")
            print(f"  Loaded Meta-Model (Logistic Regression).")
        except FileNotFoundError as e:
            print(f"Error loading artifacts: {e}. Ensure 'Training.ipynb' has been run and artifacts are in '{self.model_dir}'.")
            self.is_loaded = False
        except Exception as e:
            print(f"An unexpected error occurred during loading: {e}")
            self.is_loaded = False

    def _preprocess_input(self, input_data_raw):
        if not self.is_loaded:
            raise RuntimeError("Predictor is not loaded. Call _load_artifacts() or ensure it ran successfully in __init__.")
        
        if isinstance(input_data_raw, dict):
            input_df = pd.DataFrame([input_data_raw])
        elif isinstance(input_data_raw, pd.DataFrame):
            input_df = input_data_raw.copy()
        else:
            raise ValueError("Input data must be a dictionary (for single instance) or a Pandas DataFrame.")

        # Drop 'id' if present, as it's not a feature
        if 'id' in input_df.columns:
            input_df = input_df.drop('id', axis=1)
        # Rename 'Humidity ' if it exists
        if 'Humidity ' in input_df.columns:
            input_df.rename(columns={'Humidity ': 'Humidity'}, inplace=True)

        # Apply column encoding (Soil Type, Crop Type)
        encoded_df = self.column_encoder.transform(input_df)
        
        # Apply feature engineering
        engineered_df = feature_engineering(encoded_df)
        
        # Align features with those used during training
        final_df = pd.DataFrame(columns=self.feature_names)
        for col in self.feature_names:
            if col in engineered_df.columns:
                final_df[col] = engineered_df[col]
            else:
                final_df[col] = 0 # Or other appropriate fill for missing columns post-engineering
        return final_df[self.feature_names] # Ensure correct column order and presence

    def predict_proba_base_models(self, processed_input_df):
        """Gets averaged probabilities from each type of k-fold base model."""
        base_model_avg_probas = {}
        num_classes = len(self.target_encoder.classes_)

        # LightGBM
        lgbm_agg_proba = np.zeros((len(processed_input_df), num_classes))
        for model in self.lgbm_kfold_models:
            lgbm_agg_proba += model.predict_proba(processed_input_df)
        base_model_avg_probas['lgbm'] = lgbm_agg_proba / len(self.lgbm_kfold_models)

        # CatBoost
        catboost_agg_proba = np.zeros((len(processed_input_df), num_classes))
        for model in self.catboost_kfold_models:
            catboost_agg_proba += model.predict_proba(processed_input_df)
        base_model_avg_probas['catboost'] = catboost_agg_proba / len(self.catboost_kfold_models)

        # XGBoost
        xgb_agg_proba = np.zeros((len(processed_input_df), num_classes))
        for model in self.xgb_kfold_models:
            xgb_agg_proba += model.predict_proba(processed_input_df)
        base_model_avg_probas['xgboost'] = xgb_agg_proba / len(self.xgb_kfold_models)
        
        return base_model_avg_probas

    def predict(self, input_data_raw, top_k=1):
        if not self.is_loaded:
            print("Models not loaded. Cannot predict.")
            return None

        processed_df = self._preprocess_input(input_data_raw)
        
        # Get averaged probabilities from base models
        base_model_avg_probas = self.predict_proba_base_models(processed_df)
        
        # Stack these averaged probabilities for the meta-model
        stacked_features_for_meta = np.concatenate([
            base_model_avg_probas['lgbm'],
            base_model_avg_probas['catboost'],
            base_model_avg_probas['xgboost']
        ], axis=1)
        
        # Scale the stacked features using the loaded scaler
        stacked_features_scaled = self.stacking_scaler.transform(stacked_features_for_meta)
        
        # Predict probabilities with the meta-model
        final_pred_proba = self.meta_model.predict_proba(stacked_features_scaled)
        
        # Get top_k predictions
        if final_pred_proba.ndim == 1: # Single sample case might return 1D array from some meta-models
            final_pred_proba = final_pred_proba.reshape(1, -1)
            
        top_k_indices = np.argsort(final_pred_proba, axis=1)[:, -top_k:][:, ::-1]
        top_k_probabilities = np.array([final_pred_proba[i, top_k_indices[i]] for i in range(len(top_k_indices))])
        top_k_labels_encoded = top_k_indices
        
        # Decode labels
        results = []
        for i in range(len(top_k_labels_encoded)):
            labels = self.target_encoder.inverse_transform(top_k_labels_encoded[i])
            probs = top_k_probabilities[i]
            results.append(list(zip(labels, probs)))
        
        # If original input was a dict (single instance), return the single result directly
        if isinstance(input_data_raw, dict):
            return results[0]
        return results


## 4. Example Usage

In [ ]:
predictor = FertilizerPredictorKFold(model_dir='trained_models_kfold')

if predictor.is_loaded:
    # Example 1: Single instance prediction (as dictionary)
    sample_input_dict = {
        'Temparature': 26,
        'Humidity': 60, # Note: The training data might have had 'Humidity '
        'Moisture': 40,
        'Soil Type': 'Sandy',
        'Crop Type': 'Maize',
        'Nitrogen': 10,
        'Potassium': 15,
        'Phosphorous': 20
    }
    print("--- Predicting for a single instance (dictionary) ---")
    predictions_single = predictor.predict(sample_input_dict, top_k=3)
    print(f"Input: {sample_input_dict}")
    if predictions_single:
        print("Top 3 Predictions:")
        for label, prob in predictions_single:
            print(f"  - Fertilizer: {label}, Probability: {prob:.4f}")
    else:
        print("Prediction failed for single instance.")

    # Example 2: Batch prediction (as DataFrame)
    sample_input_df = pd.DataFrame([
        {
            'Temparature': 30, 'Humidity ': 70, 'Moisture': 50, # Intentionally using 'Humidity ' 
            'Soil Type': 'Loamy', 'Crop Type': 'Sugarcane',
            'Nitrogen': 30, 'Potassium': 25, 'Phosphorous': 40
        },
        {
            'Temparature': 22, 'Humidity': 55, 'Moisture': 35,
            'Soil Type': 'Clayey', 'Crop Type': 'Wheat',
            'Nitrogen': 15, 'Potassium': 10, 'Phosphorous': 25
        }
    ])
    print("\n--- Predicting for a DataFrame (batch) ---")
    predictions_batch = predictor.predict(sample_input_df, top_k=3)
    print(f"Input DataFrame:\n{sample_input_df}")
    if predictions_batch:
        print("Top 3 Predictions for each row:")
        for i, result_list in enumerate(predictions_batch):
            print(f"  Row {i+1}:")
            for label, prob in result_list:
                print(f"    - Fertilizer: {label}, Probability: {prob:.4f}")
    else:
        print("Prediction failed for DataFrame.")

    # Example 3: Prediction for top_k=1
    print("\n--- Predicting top 1 for single instance ---")
    predictions_top1 = predictor.predict(sample_input_dict, top_k=1)
    if predictions_top1:
        print(f"Top 1 Prediction: {predictions_top1[0][0]} (Prob: {predictions_top1[0][1]:.4f})")
    else:
        print("Top 1 prediction failed.")
else:
    print("Predictor did not load correctly. Cannot run examples.")

## 5. Notes on the Predictor

- **Artifact Dependency**: This predictor strictly depends on the artifacts generated by the `Training.ipynb` notebook, including the models saved in the `trained_models_kfold` directory.
- **Preprocessing Consistency**: It uses the exact same `DataFrameColumnEncoder` class and `feature_engineering` function definitions to ensure that input data is transformed consistently with how the models were trained.
- **K-Fold Model Averaging**: For base models (LGBM, CatBoost, XGBoost), predictions from all k-fold models are averaged to produce a robust probability estimate before feeding into the meta-model.
- **Error Handling**: Basic error handling for artifact loading is included. More sophisticated handling for new or unexpected data values during preprocessing might be needed for a production system (e.g., how to treat new categories in 'Soil Type' or 'Crop Type' not seen during training).
- **`top_k` Flexibility**: The `predict` method allows specifying `top_k` to get multiple fertilizer recommendations, which can be useful for providing alternatives.

## 6. Evaluation on Custom Labeled Dataset

This section demonstrates how to use the `FertilizerPredictorKFold` to make predictions on a custom labeled dataset and evaluate its performance.

In [ ]:
from sklearn.metrics import confusion_matrix, classification_report
import matplotlib.pyplot as plt
import seaborn as sns
import kagglehub # For loading the dataset via kagglehub

def evaluate_on_custom_dataset(predictor_instance, dataset_file_path):
    print(f"\n--- Evaluating on Custom Dataset: {dataset_file_path} ---")
    if not predictor_instance.is_loaded:
        print("Predictor is not loaded. Aborting evaluation.")
        return

    # 1. Load Custom Dataset
    try:
        custom_df_full = pd.read_csv(dataset_file_path)
        print(f"Successfully loaded custom dataset. Shape: {custom_df_full.shape}")
    except FileNotFoundError:
        print(f"Error: Custom dataset file not found at {dataset_file_path}")
        return
    except Exception as e:
        print(f"Error loading custom dataset: {e}")
        return

    if 'Fertilizer Name' not in custom_df_full.columns:
        print("Error: 'Fertilizer Name' column (truth labels) not found in the custom dataset.")
        return

    # 2. Prepare Data
    test_truth_series = custom_df_full['Fertilizer Name'].copy()
    # The input to predictor should not have the target label
    custom_df_for_prediction = custom_df_full.drop(columns=['Fertilizer Name'])
    # Also drop 'Unnamed: 0' if it exists (often an index column from saving CSVs)
    if 'Unnamed: 0' in custom_df_for_prediction.columns:
        custom_df_for_prediction = custom_df_for_prediction.drop(columns=['Unnamed: 0'])
    if 'ID' in custom_df_for_prediction.columns: # Some datasets might use 'ID'
        custom_df_for_prediction = custom_df_for_prediction.drop(columns=['ID'])

    # 3. Predict First Row
    print("\n--- Prediction for the first row ---")
    first_row_data = custom_df_for_prediction.iloc[[0]] # Pass as DataFrame to keep structure
    # The predict method returns a list of lists of (label, prob) tuples for DataFrames
    # For a single row DataFrame, it's a list containing one list of tuples.
    prediction_first_row_list = predictor_instance.predict(first_row_data, top_k=1)
    
    if prediction_first_row_list and prediction_first_row_list[0]:
        predicted_label_first_row = prediction_first_row_list[0][0][0]
        print(f"Actual Fertilizer (1st row): {test_truth_series.iloc[0]}")
        print(f"Predicted Fertilizer (1st row, top-1): {predicted_label_first_row}")
    else:
        print("Could not get prediction for the first row.")

    # 4. Predict Whole Dataset (Top-1 for CM and CSV)
    print("\n--- Predicting for the whole custom dataset (top-1) ---")
    predictions_whole_dataset_top1_list = predictor_instance.predict(custom_df_for_prediction, top_k=1)
    
    if not predictions_whole_dataset_top1_list or len(predictions_whole_dataset_top1_list) != len(custom_df_for_prediction):
        print("Error: Prediction for the whole dataset failed or returned unexpected number of results.")
        return
        
    # Extract just the predicted label string for each row
    predicted_labels_whole_dataset = [res_list[0][0] for res_list in predictions_whole_dataset_top1_list]
    print(f"Predictions generated for {len(predicted_labels_whole_dataset)} samples.")

    # 5. Display Confusion Matrix
    print("\n--- Confusion Matrix for the custom dataset ---")
    # Use target_encoder from the predictor to get all class names in the correct order
    all_class_names = predictor_instance.target_encoder.classes_
    cm = confusion_matrix(test_truth_series, predicted_labels_whole_dataset, labels=all_class_names)
    
    plt.figure(figsize=(12, 10))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Greens',
                xticklabels=all_class_names, yticklabels=all_class_names)
    plt.title('Confusion Matrix - Custom Dataset Evaluation')
    plt.xlabel('Predicted Label')
    plt.ylabel('True Label')
    plt.xticks(rotation=45, ha='right')
    plt.yticks(rotation=0)
    plt.tight_layout()
    # Save CM plot
    cm_save_path = 'custom_dataset_evaluation_cm.png'
    plt.savefig(cm_save_path)
    print(f"Confusion matrix plot saved to {cm_save_path}")
    plt.show()
    
    print("\nClassification Report:")
    print(classification_report(test_truth_series, predicted_labels_whole_dataset, labels=all_class_names, zero_division=0))

    # 6. Save Results to CSV
    print("\n--- Saving results to CSV ---")
    results_df = pd.DataFrame({
        'True_Fertilizer': test_truth_series,
        'Predicted_Fertilizer': predicted_labels_whole_dataset
    })
    # Optionally, add original features back if needed for context, but keep it simple here.
    # results_df = pd.concat([custom_df_full.reset_index(drop=True), results_df], axis=1)

    csv_save_path = 'custom_dataset_predictions_vs_truth.csv'
    results_df.to_csv(csv_save_path, index=False)
    print(f"Results saved to {csv_save_path}")
    print("\nFirst 10 samples from the CSV:")
    display(results_df.head(10))

# --- Main execution for this new section ---
if predictor.is_loaded:
    # Path to the dataset mentioned. Assuming 'Fertilizer Prediction.csv' is the target file.
    # The kagglehub path often downloads into a versioned subfolder.
    # For robustness, one might list files in the downloaded path if the exact name isn't fixed.
    try:
        # This downloads the dataset and returns the path to the directory
        dataset_dir_path = kagglehub.dataset_download("irakozekelly/fertilizer-prediction")
        # We need to find the actual CSV file within this directory.
        # Common names are 'Fertilizer Prediction.csv' or similar.
        found_csv = None
        for fname in os.listdir(dataset_dir_path):
            if fname.lower() == 'fertilizer prediction.csv' or fname.lower().endswith('.csv'): # Prioritize specific name
                found_csv = os.path.join(dataset_dir_path, fname)
                break
        if found_csv:
             evaluate_on_custom_dataset(predictor, found_csv)
        else:
            print(f"Could not find 'Fertilizer Prediction.csv' or any CSV in {dataset_dir_path}")
            print("Please ensure the file 'Fertilizer Prediction.csv' is present in the downloaded dataset folder.")

    except Exception as e:
        print(f"An error occurred trying to download or locate the dataset: {e}")
        print("Skipping evaluation on custom dataset.")
else:
    print("Predictor was not loaded. Skipping evaluation on custom dataset.")
